> Notebook-friendly copy of `part-I/1.6-geospatial-vector-data-practice-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# Practice Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.

In [2]:
%matplotlib inline
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [8]:
points = pd.DataFrame({
    "name": ["A","B"],
    "lon": [7.4, 8.5],
    "lat": [46.9, 47.4],})

stations = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points["lon"], points["lat"]),
    crs="EPSG:4326",   # WGS84 longitude/latitude in degrees
)
stations


,name,lon,lat,geometry
0,A,7.4,46.9,POINT (7.4 46.9)
1,B,8.5,47.4,POINT (8.5 47.4)


## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [7]:
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
stations.to_file("_files/points.geojson", driver="GeoJSON")
points_read = gpd.read_file("_files/points.geojson")
print(points_read.shape)
print(points_read.crs)
#has been maintained

(2, 4)
EPSG:4326


## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [14]:
import geopandas as gpd
# Read the GeoJSON file from Exercise 2
stations = gpd.read_file("_files/points.geojson")

# Reproject the stations to EPSG:2056
stations_2056 = stations.to_crs(epsg=2056)

# Print the new EPSG code
print("New EPSG code:", stations_2056.crs.to_epsg())

# Select point A
point_A = stations_2056[stations_2056["name"] == "A"]
#stations["name"] == "A" checks which rows in the name column contain the value "A".
# Print point A's projected coordinates, rounded to the nearest metre
print("Easting:", round(point_A.geometry.iloc[0].x))
print("Northing:", round(point_A.geometry.iloc[0].y))

New EPSG code: 2056
Easting: 2597056
Northing: 1194322


## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [15]:
stations_2056 = stations.to_crs(epsg=2056)
# EPSG:4326 uses longitude and latitude in degrees, not metres,
# so calculating a Euclidean distance directly would give an incorrect result.

# Select point A and point B
point_A = stations_2056[stations_2056["name"] == "A"].geometry.iloc[0]
point_B = stations_2056[stations_2056["name"] == "B"].geometry.iloc[0]

# Calculate the distance in metres and convert it to kilometres
distance_km = point_A.distance(point_B) / 1000

# Print the distance in kilometres
print("Distance between A and B:", round(distance_km, 2), "km")

Distance between A and B: 100.25 km


## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [18]:
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
in_polygon = gpd.sjoin(stations, poly, predicate="within")
# Display the points that are inside the polygon
in_polygon

,name,lon,lat,geometry,index_right,zone
0,A,7.4,46.9,POINT (7.4 46.9),0,z


## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
# Your solution here

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here